![IIT_original.png](attachment:IIT_original.png)
## <center>  Data Science Essentials (CIA2C14)</center>
### <center>  Practical 5 : Data Cleaning and Masking with Pandas </center>

In [54]:
# 1) DATA GENERATION (fair hiring rule)
from faker import Faker
import random
import pandas as pd
import numpy as np

fake = Faker()
random.seed(42)
Faker.seed(42)

# Define races and sampling weights (make 'White' 5× as likely)
races       = ['Asian', 'Black', 'White', 'Hispanic', 'Other']
race_weights = [1,       1,       7,        1,          1]

# Build dataset
records = []
for _ in range(1000):
    name = fake.name()
    age  = random.randint(22, 60)
    # sample race with weights
    race = random.choices(races, weights=race_weights, k=1)[0]

    gpa = round(random.uniform(2.0, 4.0), 2)
    # years_of_exp between 0 and (age - 21)
    years_of_exp = random.randint(0, max(age - 21, 0))

    # Fair hiring rule: require GPA ≥ 3.0 AND at least 5 years experience
    hired = int((gpa >= 3.0) and (years_of_exp >= 5))

    records.append([name, age, race, gpa, years_of_exp, hired])

df = pd.DataFrame(records, columns=['name','age','race','gpa','years_of_exp','hired'])
print(df['race'].value_counts(normalize=True))  # check proportions
print(df.head())

# Save to CSV
df.to_csv('hiring_data.csv', index=False)


race
White       0.631
Other       0.101
Black       0.092
Hispanic    0.092
Asian       0.084
Name: proportion, dtype: float64
              name  age   race   gpa  years_of_exp  hired
0     Allison Hill   29  Asian  2.55             3      0
1      Noah Rhodes   30  White  3.35             8      1
2  Angie Henderson   27  White  2.06             0      0
3    Daniel Wagner   35  White  3.20             8      1
4  Cristian Santos   34  White  3.40             6      1


In [55]:
# 2) “UNFAIR ASSUMPTION” MODEL: include race as a feature
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.linear_model    import LogisticRegression
from sklearn.metrics         import classification_report, confusion_matrix
import random
import pandas as pd
import numpy as np
#load
df = pd.read_csv('hiring_data.csv')

# Features & label
X = df[['race','gpa','years_of_exp','age']]
y = df['hired']

df




,name,age,race,gpa,years_of_exp,hired
0,Allison Hill,29,Asian,2.55,3,0
1,Noah Rhodes,30,White,3.35,8,1
2,Angie Henderson,27,White,2.06,0,0
3,Daniel Wagner,35,White,3.20,8,1
4,Cristian Santos,34,White,3.40,6,1
...,...,...,...,...,...,...
995,Brittany Ward,30,Other,2.01,0,0
996,Edward Stanley,57,Other,3.68,30,1
997,Christina Johnson,54,White,3.82,11,1
998,Edgar Miller,30,Black,3.32,9,1


In [59]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

#encode manually
race_mapping = {'Asian': 1, 'Black': 2, 'Hispanic': 3, 'White': 4, 'Other': -1}
X_train['race'] = X_train['race'].map(race_mapping)
X_test['race'] = X_test['race'].map(race_mapping)

#scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#train
clf_pi = LogisticRegression()
clf_pi.fit(X_train, y_train)

#predict
y_pred = clf_pi.predict(X_test)

In [60]:
import math

feature_names = X.columns

intercept = clf_pi.intercept_[0]
coefs     = clf_pi.coef_[0]

# Build the linear part: β0 + β1 x1 + β2 x2 + ...
terms = [f"{intercept:.3f}"]
for β, name in zip(coefs, feature_names):
    terms.append(f"{β:.3f}*{name}")

linear_eq = " + ".join(terms)

print(f"logit(p) = {linear_eq}")
print(f"p = 1 / (1 + exp(-({linear_eq})))")


logit(p) = -1.910 + -0.041*race + 2.586*gpa + 1.614*years_of_exp + -0.038*age
p = 1 / (1 + exp(-(-1.910 + -0.041*race + 2.586*gpa + 1.614*years_of_exp + -0.038*age)))


In [61]:
#feature importance
feature_importances = clf_pi.coef_[0]
for feature, importance in zip(feature_names, feature_importances):
    print(f"{feature}: {importance:.3f}")

race: -0.041
gpa: 2.586
years_of_exp: 1.614
age: -0.038
